## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, lower, to_timestamp, to_date, when, length

## Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_orders")

## Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Status normalization

In [0]:
df = df.withColumn("order_status", lower(col("order_status")))

### Cleaning Dates

In [0]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_timestamp"
]

for c in timestamp_cols:
    df = df.withColumn(
        c,
        when(
            col(c).isNull(),
            None
        ).otherwise(
            to_timestamp(col(c))
        )
    )

In [0]:
df = df.withColumn(
    "order_estimated_delivery_date",
    when(
        col("order_estimated_delivery_date").isNull(),
        None
    ).otherwise(
        to_date(col("order_estimated_delivery_date"))
    )
)

### Renaming Columns

In [0]:
RENAME_MAP = {
    "order_purchase_timestamp": "purchase_timestamp",
    "order_approved_at": "approved_at",
    "order_delivered_timestamp": "delivered_timestamp",
    "order_estimated_delivery_date": "estimated_delivery_date",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity checks of dataframe

In [0]:
df.limit(10).display()

## Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_orders")

### Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_orders LIMIT 10